# 🚀 FAKTA — LSTM Training di Google Colab

Notebook ini untuk training model LSTM di Google Colab dengan GPU gratis.

## Cara Pakai:
1. Buka https://colab.research.google.com/
2. Klik **Upload** → pilih file ini
3. Runtime → Change runtime type → **T4 GPU**
4. Run semua cell dari atas ke bawah
5. Download model yang sudah di-train dari sidebar Files

---
## Step 1: Cek GPU & Setup Environment

In [ ]:
# Cek apakah GPU tersedia
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("GPU name:", tf.test.gpu_device_name())

# Install dependencies tambahan
!pip install -q scikit-learn pandas numpy matplotlib seaborn pyyaml

---
## Step 2: Upload Dataset

Ada 2 cara:

**Cara A — Upload manual:**
1. Klik ikon 📁 di sidebar kiri
2. Upload file CSV ke folder `data/training/`

**Cara B — Download dari internet (langsung dari notebook):**

In [ ]:
# Buat folder structure
!mkdir -p data/training data/processed models/lstm

# ===== PILIH SALAH SATU CARA =====

# CARA A: Upload manual dari laptop Anda
from google.colab import files
print("Upload file CSV dataset Anda (hoax/valid/uncertain)...")
uploaded = files.upload()
for filename in uploaded.keys():
    !mv {filename} data/training/
    print(f"Moved {filename} to data/training/")

# CARA B: Download dataset dari URL (uncomment jika mau pakai ini)
# !wget -O data/training/dataset.csv "URL_DATASET_ANDA"

print("\nDataset siap!")
!ls -la data/training/

---
## Step 3: Load & Preprocess Dataset

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Load semua CSV di folder training
csv_files = list(Path('data/training').glob('*.csv'))
print(f"Found {len(csv_files)} CSV files")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    # Auto-detect column names
    if 'text' not in df.columns:
        for col in ['tweet', 'content', 'article', 'berita']:
            if col in df.columns:
                df = df.rename(columns={col: 'text'})
                break
    if 'label' not in df.columns:
        for col in ['hoax', 'class', 'is_hoax', 'label']:
            if col in df.columns:
                df = df.rename(columns={col: 'label'})
                break
    if 'text' in df.columns and 'label' in df.columns:
        df['label'] = df['label'].astype(str).str.lower().str.strip()
        df['label'] = df['label'].map({
            'hoax': 'hoax', 'true': 'valid', 'false': 'hoax',
            '1': 'hoax', '0': 'valid', 'hoaks': 'hoax',
            'valid': 'valid', 'tidak hoax': 'valid', 'non-hoax': 'valid',
            'uncertain': 'uncertain', 'netral': 'uncertain', 'unknown': 'uncertain',
        }).fillna(df['label'])
        df = df[df['label'].isin(['hoax', 'valid', 'uncertain'])]
        dfs.append(df[['text', 'label']])
        print(f"  {f.name}: {len(df)} samples")

if not dfs:
    raise ValueError("No valid CSV files found! Upload dataset first.")

combined = pd.concat(dfs, ignore_index=True)
combined = combined.dropna(subset=['text', 'label'])
combined = combined.drop_duplicates(subset=['text'])
combined = combined[combined['text'].str.len() > 10]

print(f"\nTotal dataset: {len(combined)} samples")
print(f"Label distribution:")
print(combined['label'].value_counts())

---
## Step 4: Preprocessing Text

In [ ]:
import re

# Simple Indonesian slang dictionary
SLANG_MAP = {
    'yg': 'yang', 'dg': 'dengan', 'dr': 'dari', 'tp': 'tapi',
    'klo': 'kalau', 'kalo': 'kalau', 'gak': 'tidak', 'ga': 'tidak',
    'nggak': 'tidak', 'ngga': 'tidak', 'udh': 'sudah', 'udah': 'sudah',
    'blm': 'belum', 'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'skrg': 'sekarang', 'bs': 'bisa', 'krn': 'karena',
    'utk': 'untuk', 'dpt': 'dapat', 'sm': 'sama', 'jg': 'juga',
    'cuma': 'hanya', 'cm': 'hanya', 'aja': 'saja',
    'sy': 'saya', 'gw': 'saya', 'gue': 'saya',
    'lu': 'kamu', 'loe': 'kamu', 'km': 'kamu',
    'mrk': 'mereka', 'tdk': 'tidak', 'kpn': 'kapan',
    'nih': 'ini', 'tu': 'itu', 'gitu': 'begitu',
    'sih': '', 'deh': '', 'dong': '', 'lah': '',
}

def clean_text(text):
    """Clean and normalize Indonesian text."""
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove special chars (keep basic punctuation)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"()-]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_slang(text):
    """Replace slang with formal Indonesian."""
    words = text.split()
    normalized = [SLANG_MAP.get(w, w) for w in words]
    return ' '.join(normalized)

# Apply preprocessing
print("Preprocessing text...")
combined['cleaned'] = combined['text'].apply(clean_text)
combined['normalized'] = combined['cleaned'].apply(normalize_slang)

print(f"Sample cleaned text:")
print(combined['normalized'].iloc[0][:200])

---
## Step 5: Tokenize & Split Data

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import pickle

# Config
MAX_WORDS = 20000   # Vocabulary size
MAX_LEN = 200       # Max sequence length

# Tokenize
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(combined['normalized'].tolist())

sequences = tokenizer.texts_to_sequences(combined['normalized'].tolist())
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Encode labels
label_map = {'valid': 0, 'hoax': 1, 'uncertain': 2}
y = np.array([label_map[l] for l in combined['label']])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

# Split: 70% train, 15% val, 15% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.18, random_state=42, stratify=y_train
)

print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Class weights (handle imbalance)
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print(f"Class weights: {class_weight}")

---
## Step 6: Build BiLSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Model config
EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.3
NUM_CLASSES = 3

# Build model
model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN,
        mask_zero=True,
    ),
    Dropout(DROPOUT_RATE),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True, dropout=DROPOUT_RATE)),
    Bidirectional(LSTM(LSTM_UNITS // 2, return_sequences=False)),
    Dense(32, activation='relu'),
    Dropout(DROPOUT_RATE),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

---
## Step 7: Train Model

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('models/lstm/lstm_model.keras', save_best_only=True, monitor='val_loss', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

# Train
EPOCHS = 20
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
)

---
## Step 8: Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print("Training history saved to training_history.png")

---
## Step 9: Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Test evaluation
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n{'='*50}")
print(f"TEST SET EVALUATION")
print(f"{'='*50}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Loss: {loss:.4f}")

# Classification report
y_pred = np.argmax(model.predict(X_test), axis=1)
labels = ['valid', 'hoax', 'uncertain']
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=labels))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix saved to confusion_matrix.png")

---
## Step 10: Quick Inference Test

In [ ]:
def predict(text):
    """Predict hoax probability for a single text."""
    # Clean
    cleaned = clean_text(text)
    normalized = normalize_slang(cleaned)
    # Tokenize & pad
    seq = tokenizer.texts_to_sequences([normalized])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    # Predict
    probs = model.predict(padded, verbose=0)[0]
    return {
        'hoax': float(probs[1]),
        'valid': float(probs[0]),
        'uncertain': float(probs[2]),
    }

# Test samples
test_texts = [
    ("VIRAL!!! Matcha menyebabkan gagal ginjal!! Sebarkan sebelum dihapus!!", "hoax"),
    ("BMKG mencatat gempa magnitudo 5.2 di Maluku pada 15 Januari 2025.", "valid"),
    ("Kabar beredar BBM naik, belum ada konfirmasi resmi.", "uncertain"),
]

print(f"\n{'='*50}")
print(f"INFERENCE TEST")
print(f"{'='*50}")

for text, expected in test_texts:
    result = predict(text)
    verdict = max(result, key=result.get)
    print(f"\nText: {text[:60]}...")
    print(f"  Expected: {expected}")
    print(f"  Predicted: {verdict.upper()}")
    print(f"  Hoax: {result['hoax']:.4f} | Valid: {result['valid']:.4f} | Uncertain: {result['uncertain']:.4f}")
    correct = "✅" if verdict == expected else "❌"
    print(f"  {correct}")

---
## Step 11: Save Model & Tokenizer

In [ ]:
# Save tokenizer
with open('models/lstm/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save label map
import json
with open('models/lstm/label_map.json', 'w') as f:
    json.dump(label_map, f)

print("Tokenizer saved to models/lstm/tokenizer.pkl")
print("Label map saved to models/lstm/label_map.json")
print("Model saved to models/lstm/lstm_model.keras")

---
## Step 12: Download Model untuk Dipakai di Lokal

Run cell ini untuk download semua file model ke laptop Anda.
File akan otomatis ter-download sebagai ZIP.

In [ ]:
from google.colab import files
import zipfile
import os

# Zip model files
with zipfile.ZipFile('lstm_model.zip', 'w') as zf:
    zf.write('models/lstm/lstm_model.keras')
    zf.write('models/lstm/tokenizer.pkl')
    zf.write('models/lstm/label_map.json')
    zf.write('training_history.png')
    zf.write('confusion_matrix.png')

print("Downloading model package...")
files.download('lstm_model.zip')
print("\nDone! Extract lstm_model.zip dan taruh di folder FAKTA/models/lstm/")